In [1]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F


In [2]:
Z_train = np.load("../artifacts/data/Z_train.npy")
Z_test  = np.load("../artifacts/data/Z_test.npy")

y_train = np.load("../artifacts/data/y_train.npy")
y_test  = np.load("../artifacts/data/y_test.npy")

print(Z_train.shape, Z_test.shape)



(175341, 64) (82332, 64)


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

Z_train_t = torch.tensor(Z_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)

Z_test_t  = torch.tensor(Z_test, dtype=torch.float32).to(device)
y_test_t  = torch.tensor(y_test, dtype=torch.float32).to(device)


In [4]:
class EmbeddingClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze()


In [5]:
input_dim = Z_train.shape[1]
model = EmbeddingClassifier(input_dim).to(device)

model.load_state_dict(
    torch.load("../artifacts/models/classifier_embeddings.pth",
               map_location=device)
)

model.train()


EmbeddingClassifier(
  (net): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [6]:
#nesta etapa quero pertubar o modelo com dados para que o mesmo erre
#na vida real, hackers podem tentar adicionar pacotes para alterar as características
#do tráfego (194 campos originais) de modo que, após passar pelo encode, o ponto resultante do espaço 
#de 64 dimensões cruze a fronteira de decisão e escape da detecção
#FGSM é um ataque de passo único que utiliza o gradiente da rede para encontrar
#a direção que mais aumenta o erro do modelo

def fgsm_attack(x, y, model, criterion, epsilon):
    x_adv = x.clone().detach().requires_grad_(True)
    logits = model(x_adv).squeeze()
    loss = criterion(logits, y)
    loss.backward()
    perturbation = epsilon * x_adv.grad.sign()
    return (x_adv + perturbation).detach()


In [7]:
# NOVA CÉLULA: Definição do PGD
def generate_pgd_attacks(model, x, y, epsilon=0.05, alpha=0.01, iterations=10):
    x_adv = x.clone().detach().requires_grad_(True)
    
    for i in range(iterations):
        outputs = model(x_adv)
        loss = F.cross_entropy(outputs, y)
        loss.backward()
        
        with torch.no_grad():
            x_adv = x_adv + alpha * x_adv.grad.sign()
            delta = torch.clamp(x_adv - x, min=-epsilon, max=epsilon)
            x_adv = torch.clamp(x + delta, min=0, max=1).detach().requires_grad_(True)
            
    return x_adv

In [8]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

epsilon = 0.02
epochs = 5

for epoch in range(epochs):
    model.train()

    # dados limpos
    logits = model(Z_train_t).squeeze()
    loss_clean = criterion(logits, y_train_t)

    # dados adversariais
    Z_adv = fgsm_attack(Z_train_t, y_train_t, model, criterion, epsilon)
    logits_adv = model(Z_adv).squeeze()
    loss_adv = criterion(logits_adv, y_train_t)

    # loss combinada
    loss = 0.5 * loss_clean + 0.5 * loss_adv

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1} - Loss clean: {loss_clean.item():.4f} | Loss adv: {loss_adv.item():.4f}")


Epoch 1 - Loss clean: 0.1142 | Loss adv: 0.2220
Epoch 2 - Loss clean: 0.1136 | Loss adv: 0.2192
Epoch 3 - Loss clean: 0.1130 | Loss adv: 0.2165
Epoch 4 - Loss clean: 0.1125 | Loss adv: 0.2140
Epoch 5 - Loss clean: 0.1120 | Loss adv: 0.2116


In [9]:
model.eval()

# 1. Avaliação Normal (Sem ataque)
with torch.no_grad():
    probs_clean = torch.sigmoid(model(Z_test_t)).cpu().numpy()
auc_clean = roc_auc_score(y_test, probs_clean)

# 2. Avaliação sob Ataque (Usando a função PGD que você criou)
# Criamos uma versão "perturbada" do conjunto de teste
Z_test_adv = generate_pgd_attacks(model, Z_test_t, torch.FloatTensor(y_test).to(Z_test_t.device), 
                                  epsilon=0.05, alpha=0.01, iterations=10)

with torch.no_grad():
    probs_adv = torch.sigmoid(model(Z_test_adv)).cpu().numpy()
auc_adv = roc_auc_score(y_test, probs_adv)

print(f"ROC AUC (Tráfego Normal): {auc_clean:.4f}")
print(f"ROC AUC (Sob Ataque PGD): {auc_adv:.4f}")

# O objetivo é que o auc_adv não seja muito menor que o auc_clean

ROC AUC (Tráfego Normal): 0.9741
ROC AUC (Sob Ataque PGD): 0.6892


In [10]:
torch.save(
    model.state_dict(),
    "../artifacts/models/classifier_embeddings_adversarial.pth"
)
